# Multimodal LLMs — scratch থেকে একটি toy CLIP-style contrastive training loop

এটি কোনো বাস্তব image, text বা pretrained weights download করে **না** (internet access আছে বলে ধরে নেওয়া হয়নি)। বরং এটি একটি সম্পূর্ণ synthetic toy সমস্যা তৈরি করে, যা সেই একটি ধারণাকে ধরে রাখে যেটি সমাধান করার জন্যই CLIP-এর loss আসলে তৈরি:

- `NUM_CATEGORIES` টি লুকানো "concept" আছে (বাস্তব জগতের category যেমন "cat", "beach", "mountain", ...-এর stand-in)। প্রতিটি category-র নিজস্ব একটি random latent vector আছে।
- প্রতিটি paired training example-এর জন্য, আমরা একটি category sample করি, তারপর একটি "image" feature vector এবং একটি "text" feature vector তৈরি করি যা **দুটোই** সেই একই category latent থেকে আসে, কিন্তু ভিন্ন, noisy, modality-specific transformation-এর মধ্য দিয়ে — ঠিক যেমন একটি বিড়ালের বাস্তব ছবি এবং "cat" শব্দটি একটি অন্তর্নিহিত concept ভাগ করে, কিন্তু raw-pixel / raw-token স্তরে একেবারেই একরকম দেখায় না।
- দুটি ছোট encoder MLP (প্রতি modality-তে একটি) README-এর symmetric InfoNCE contrastive loss দিয়ে এই synthetic pair-গুলোর batch-এর উপর train করা হয়, category label-এ **কোনো** access ছাড়াই — শুধু এই তথ্যটুকু যে image i এবং text i একসাথে paired ছিল।
- তারপর আমরা cross-modal retrieval accuracy মাপি (একটি image দেওয়া হলে, কয়েকটি candidate-এর মধ্যে তার মিলিত text খুঁজে বের করা) training-এর **আগে** (random encoder weights) এবং training-এর **পরে**, এমন একটি held-out set-এ যা training-এর সময় কখনো দেখা হয়নি, যাতে একটি প্রকৃত, মাপা উন্নতি দেখানো যায়।

Runtime: CPU-তে কয়েক সেকেন্ড (ছোট MLP, 400 training step)।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান। সেকশন 1-5 কেবল data, model, loss, evaluation ও training loop সংজ্ঞায়িত করে; পুরো experiment শেষ সেকশনের `main()`-এ চলে।

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## 1. Synthetic paired (image, text) data তৈরি

এখানে সব global constant (category সংখ্যা, latent dimension, raw image/text feature dimension, shared embedding dimension) সংজ্ঞায়িত হয়, সাথে প্রতিটি modality-র জন্য একটি fixed random "rendering" linear map এবং প্রতিটি category-র জন্য একটি random latent vector। `sample_pairs` একই category latent থেকে (image_raw, text_raw) জোড়া তৈরি করে, ভিন্ন rendering ও স্বাধীন noise দিয়ে — ফলে দুটি vector signal ভাগ করে, কিন্তু অভিন্ন নয়।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Synthetic paired (image, text) data তৈরি
# ---------------------------------------------------------------------------

NUM_CATEGORIES = 10        # "cat", "beach", ...-এর মতো বাস্তব concept-এর stand-in
LATENT_DIM = 8             # shared অন্তর্নিহিত concept-এর dimensionality
IMAGE_RAW_DIM = 24         # encoding-এর আগে "raw" image-feature dimensionality
TEXT_RAW_DIM = 20          # encoding-এর আগে "raw" text-feature dimensionality
EMBED_DIM = 16             # দুটি encoder যে shared embedding dimension-এ map করে

# Fixed random "rendering" transform: প্রতিটি modality একটি category-র shared
# latent-কে একটি ভিন্ন random linear map-এর মাধ্যমে নিজস্ব raw feature space-এ
# রূপান্তর করে, তাই একই category-র একটি image-vector ও একটি text-vector signal
# ভাগ করে কিন্তু মোটেও অভিন্ন নয় -- অনেকটা যেমন একটি কুকুরের ছবি এবং "dog"
# string একটি concept ভাগ করে কিন্তু একটি representation নয়।
image_render = torch.randn(LATENT_DIM, IMAGE_RAW_DIM)
text_render = torch.randn(LATENT_DIM, TEXT_RAW_DIM)

# প্রতিটি category-র জন্য একটি random latent vector, পুরো script জুড়ে fixed।
category_latents = torch.randn(NUM_CATEGORIES, LATENT_DIM)


def sample_pairs(n_pairs, noise_std=0.6):
    """n_pairs সংখ্যক (image_raw, text_raw, category_id) sample করে।

    একই pair-এর image ও text raw vector একই category latent থেকে তৈরি হয়,
    তারপর ভিন্ন modality-specific linear rendering-এর মধ্য দিয়ে পাঠানো হয় এবং
    স্বাধীন noise দেওয়া হয় -- তাই তারা signal (category) ভাগ করে কিন্তু
    তুচ্ছভাবে অভিন্ন vector নয়।
    """
    category_ids = torch.randint(0, NUM_CATEGORIES, (n_pairs,))
    latents = category_latents[category_ids]                      # (n_pairs, LATENT_DIM)

    image_raw = latents @ image_render + noise_std * torch.randn(n_pairs, IMAGE_RAW_DIM)
    text_raw = latents @ text_render + noise_std * torch.randn(n_pairs, TEXT_RAW_DIM)
    return image_raw, text_raw, category_ids

## 2. দুটি encoder (প্রতি modality-তে একটি ছোট MLP)

`Encoder` একটি modality-র raw feature vector-কে shared embedding space-এ map করে: Linear → GELU → Linear, তারপর output-কে unit length-এ normalize করা হয় (README-এর মতো), যাতে dot product সরাসরি cosine similarity হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 2. দুটি encoder (প্রতি modality-তে একটি ছোট MLP)
# ---------------------------------------------------------------------------

class Encoder(nn.Module):
    """একটি modality-র raw feature vector-কে shared embedding space-এ map করে।"""

    def __init__(self, in_dim, embed_dim, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim),
        )

    def forward(self, x):
        out = self.net(x)
        return F.normalize(out, dim=-1)   # unit-length embedding, README-এর মতো

## 3. Symmetric CLIP / InfoNCE contrastive loss

একটি batch-এর উপর symmetric InfoNCE loss: `logits[i, j]` হলো image i ও text j-এর cosine similarity গুণ `exp(log_temperature)`। যেহেতু batch-গুলো paired, row i (এবং column i)-এর সঠিক match সবসময় index i। Image→text এবং text→image দুই দিকের cross-entropy-র গড় নেওয়া হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 3. Symmetric CLIP / InfoNCE contrastive loss
# ---------------------------------------------------------------------------

def clip_contrastive_loss(image_embeds, text_embeds, log_temperature):
    """একটি batch-এর উপর symmetric InfoNCE loss, ঠিক README-তে যেমন বর্ণিত।

    logits[i, j] = cosine_similarity(image_i, text_j) * exp(log_temperature)
    Row i (এবং column i)-এর সঠিক match সবসময় index i, যেহেতু image_embeds
    এবং text_embeds paired batch হিসেবে তৈরি।
    """
    batch_size = image_embeds.shape[0]
    logits = (image_embeds @ text_embeds.T) * log_temperature.exp()
    labels = torch.arange(batch_size)

    loss_i2t = F.cross_entropy(logits, labels)          # প্রতিটি image তার text বেছে নেয়
    loss_t2i = F.cross_entropy(logits.T, labels)         # প্রতিটি text তার image বেছে নেয়
    return (loss_i2t + loss_t2i) / 2

## 4. Retrieval evaluation: একটি image দেওয়া হলে, তার মিলিত text খুঁজে বের করা

প্রতিটি query image-এর জন্য `n_candidates` টি text-এর একটি candidate pool তৈরি হয় (1টি সত্যিকারের match + অন্য pair-গুলো থেকে বাকি distractor), এবং nearest-neighbor cosine similarity সত্যিকারের match-টি বেছে নেয় কিনা তা যাচাই করা হয় — বাস্তব cross-modal retrieval-এর প্রতিফলন: "এই যে একটি image, এই caption-গুলোর কোনটি আসলে এটি বর্ণনা করে?"

In [ ]:
# ---------------------------------------------------------------------------
# 4. Retrieval evaluation: একটি image দেওয়া হলে, তার মিলিত text খুঁজে বের করা
# ---------------------------------------------------------------------------

@torch.no_grad()
def retrieval_accuracy(image_encoder, text_encoder, n_queries=200, n_candidates=8):
    """প্রতিটি query image-এর জন্য n_candidates টি text-এর একটি candidate pool
    তৈরি করো (1টি সত্যিকারের match + অন্য pair-গুলো থেকে (n_candidates - 1) টি
    distractor) এবং যাচাই করো nearest-neighbor cosine similarity সত্যিকারের
    match-টি বেছে নেয় কিনা।

    এটি বাস্তব cross-modal retrieval-এর প্রতিফলন: "এই যে একটি image, এই
    caption-গুলোর কোনটি আসলে এটি বর্ণনা করে?"
    """
    image_raw, text_raw, _ = sample_pairs(n_queries)
    image_embeds = image_encoder(image_raw)                  # (n_queries, EMBED_DIM)
    text_embeds = text_encoder(text_raw)                      # (n_queries, EMBED_DIM), text_embeds[i] মেলে image_embeds[i]-এর সাথে

    correct = 0
    for i in range(n_queries):
        # Candidate pool: সত্যিকারের মিলিত text, সাথে অন্য query-গুলোর
        # (সম্পর্কহীন) paired text থেকে নেওয়া distractor text।
        distractor_pool = [j for j in range(n_queries) if j != i]
        distractor_idx = torch.tensor(
            distractor_pool[:n_candidates - 1]
            if len(distractor_pool) >= n_candidates - 1
            else distractor_pool
        )
        candidate_idx = torch.cat([torch.tensor([i]), distractor_idx])
        candidates = text_embeds[candidate_idx]               # (n_candidates, EMBED_DIM)

        sims = image_embeds[i] @ candidates.T                  # cosine sim (ইতিমধ্যে unit-norm)
        predicted = sims.argmax().item()
        if predicted == 0:                                      # candidate_idx-এর index 0 সবসময় সত্যিকারের match
            correct += 1

    return correct / n_queries

## 5. Training loop

`train_clip_style` দুটি encoder-এর parameter এবং একটি learned `log_temperature` (শুরুতে `exp(0)=1`) একসাথে Adam দিয়ে optimize করে। প্রতি step-এ একটি নতুন synthetic batch sample করা হয়, contrastive loss গণনা করা হয়, এবং প্রথম step ও প্রতি 100 step পরপর loss ছাপা হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 5. Training loop
# ---------------------------------------------------------------------------

def train_clip_style(image_encoder, text_encoder, steps=400, batch_size=32, lr=1e-2):
    log_temperature = torch.tensor(0.0, requires_grad=True)   # learned temperature, শুরু exp(0)=1 থেকে
    params = list(image_encoder.parameters()) + list(text_encoder.parameters()) + [log_temperature]
    optimizer = torch.optim.Adam(params, lr=lr)

    losses = []
    for step in range(steps):
        image_raw, text_raw, _ = sample_pairs(batch_size)
        image_embeds = image_encoder(image_raw)
        text_embeds = text_encoder(text_raw)

        loss = clip_contrastive_loss(image_embeds, text_embeds, log_temperature)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

        if step == 0 or (step + 1) % 100 == 0:
            print(f"  step {step + 1:4d}/{steps}   contrastive loss = {loss.item():.4f}")

    return losses

## 6. Main script: পুরো experiment একসাথে

এই script-এ আলাদা demo function নেই — পুরো experiment-টিই `main()`: encoder তৈরি, training-এর **আগে** retrieval accuracy মাপা (chance level-এর সাথে তুলনা), contrastive loss দিয়ে training, training-এর **পরে** আবার accuracy মাপা, এবং শেষে একটি একক image-এর জন্য nearest-neighbor retrieval-এর একটি পূর্ণ worked example ছাপা। তাই এখানে `main()` সরাসরি একবার চালানো হয় (demo দুবার চলার কোনো সমস্যা নেই)।

In [ ]:
# ---------------------------------------------------------------------------
# 6. Main script
# ---------------------------------------------------------------------------

def main():
    print("=" * 70)
    print("TOY CLIP: ALIGNING TWO MODALITIES WITH A CONTRASTIVE LOSS")
    print("=" * 70)
    print(f"{NUM_CATEGORIES} hidden categories, image_raw_dim={IMAGE_RAW_DIM}, "
          f"text_raw_dim={TEXT_RAW_DIM}, shared embed_dim={EMBED_DIM}")
    print("Image and text vectors for the same pair share a category latent")
    print("but are rendered through different random linear maps plus noise --")
    print("they are correlated, not identical, just like a photo and a caption.\n")

    image_encoder = Encoder(IMAGE_RAW_DIM, EMBED_DIM)
    text_encoder = Encoder(TEXT_RAW_DIM, EMBED_DIM)

    # একটি random-batch sanity check: random raw feature দিয়ে (কোনো shared
    # category structure ছাড়াই), একটি random encoder মোটামুটি chance level-এ
    # retrieve করা উচিত। আমরা বরং নিচে training-এর আগে ও পরে দুবারই প্রকৃত
    # data-generating process-এর উপর evaluate করি, যা README-এর প্রতিশ্রুত
    # ন্যায্য, সৎ তুলনা।
    n_candidates = 8
    chance_level = 1.0 / n_candidates

    print("-" * 70)
    print("RETRIEVAL ACCURACY BEFORE TRAINING (randomly initialized encoders)")
    print("-" * 70)
    acc_before = retrieval_accuracy(image_encoder, text_encoder, n_candidates=n_candidates)
    print(f"Task: given an image, pick its matching text among {n_candidates} candidates.")
    print(f"Chance level (uniform random guess): {chance_level:.3f}")
    print(f"Measured accuracy BEFORE training:    {acc_before:.3f}")
    print("-> With random encoder weights, accuracy is already somewhat above chance")
    print("   (not exactly at it): the raw image and text vectors are genuinely")
    print("   correlated by construction (same category latent), and random linear")
    print("   projections do not fully destroy that correlation. But the encoders")
    print("   have not been trained to EXPLOIT it yet -- watch this number rise")
    print("   substantially once the contrastive loss is applied below.\n")

    print("-" * 70)
    print("TRAINING WITH THE SYMMETRIC CONTRASTIVE (INFONCE) LOSS")
    print("-" * 70)
    losses = train_clip_style(image_encoder, text_encoder)
    print(f"\nLoss went from {losses[0]:.4f} (step 1) to {losses[-1]:.4f} (final step).")

    print("\n" + "-" * 70)
    print("RETRIEVAL ACCURACY AFTER TRAINING")
    print("-" * 70)
    acc_after = retrieval_accuracy(image_encoder, text_encoder, n_candidates=n_candidates)
    print(f"Measured accuracy AFTER training:     {acc_after:.3f}")
    print(f"(same task: 1 true match among {n_candidates} candidates, chance = {chance_level:.3f})")

    improvement = acc_after - acc_before
    print(f"\n-> Retrieval accuracy improved by {improvement:+.3f} after training")
    print("   (from {:.3f} to {:.3f}). The contrastive loss pulled each image's".format(acc_before, acc_after))
    print("   embedding toward its own paired text's embedding and away from")
    print("   unrelated texts in the batch -- with no labels beyond 'these two")
    print("   were paired' -- which is exactly the CLIP recipe from the README,")
    print("   just at toy scale with synthetic vectors instead of real images")
    print("   and captions.")

    # -----------------------------------------------------------------
    # Zero-shot-style demo: কয়েকটি candidate-এর মধ্য থেকে একটি নতুন image-এর
    # জন্য সঠিক text বেছে নেওয়া, intuition-এর জন্য পুরোটা ছাপা।
    # -----------------------------------------------------------------
    print("\n" + "=" * 70)
    print("ONE WORKED EXAMPLE: NEAREST-NEIGHBOR RETRIEVAL FOR A SINGLE IMAGE")
    print("=" * 70)
    with torch.no_grad():
        image_raw, text_raw, category_ids = sample_pairs(6)
        image_embeds = image_encoder(image_raw)
        text_embeds = text_encoder(text_raw)

        query_idx = 0
        sims = image_embeds[query_idx] @ text_embeds.T   # সব 6টি text-এর সাথে similarity
        ranked = sims.argsort(descending=True).tolist()

        print(f"Query image belongs to hidden category {category_ids[query_idx].item()}")
        print(f"Candidate texts belong to categories:   {category_ids.tolist()}")
        print(f"Cosine similarities to each candidate:  {sims.numpy().round(3)}")
        print(f"Ranked candidate indices (best first):  {ranked}")
        top_pick_category = category_ids[ranked[0]].item()
        is_correct = ranked[0] == query_idx
        print(f"Top-ranked candidate is index {ranked[0]} (category {top_pick_category}), "
              f"true match is index {query_idx} (category {category_ids[query_idx].item()})")
        print(f"-> Top-1 retrieval on this single worked example is "
              f"{'CORRECT' if is_correct else 'INCORRECT'}.")


main()